In [1]:
# Install GPU-enabled PyTorch and YOLOv8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install ultralytics matplotlib
print("✅ Installation complete!")

Looking in indexes: https://download.pytorch.org/whl/cu124
✅ Installation complete!


In [2]:
import torch
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt

# Check for RTX 2050
device = 0 if torch.cuda.is_available() else 'cpu'
if device == 0:
    print(f"✅ Success! Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU not detected. Please update NVIDIA drivers to 525+.")
print(f"Device: {device}")

✅ Success! Using GPU: NVIDIA GeForce RTX 2050
Device: 0


In [3]:
# UPDATE THIS CELL IN YOUR NOTEBOOK
dataset_path = r"C:\Users\revan\PyCharmMiscProject\Project\Tomato-Leaf-Disease-63"

data_yaml = f"""
train: {dataset_path}/train/images
val: {dataset_path}/valid/images
test: {dataset_path}/test/images

nc: 9
names: [
  'Early_Blight', 'Healthy', 'Late_Blight', 'Leaf_Miner',
  'Leaf_Mold', 'Mosaic_Virus', 'Septoria_Leaf_Spot',
  'Spider_Mites', 'Yellow_Leaf_Curl_Virus'
]
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)
print("✅ Data YAML created successfully!")
print(f"Dataset path: {dataset_path}")
print(f"Dataset exists: {os.path.exists(dataset_path)}")

✅ Data YAML created successfully!
Dataset path: C:\Users\revan\PyCharmMiscProject\Project\Tomato-Leaf-Disease-63
Dataset exists: True


In [ ]:
# Auto-detect the latest training run folder (YOLO increments: notebook, notebook2, notebook3...)
from pathlib import Path
# Use absolute path so it works regardless of notebook working directory
project_dir = Path(r'C:\Users\revan\PyCharmMiscProject\Project')
run_dir = project_dir / 'runs' / 'detect'
print(f"🔍 Looking for runs in: {run_dir}")
print(f"Run dir exists: {run_dir.exists()}")
matching = sorted(run_dir.glob('tomato_disease_notebook*'), key=os.path.getmtime) if run_dir.exists() else []
latest_run = str(matching[-1]) if matching else str(run_dir / 'tomato_disease_notebook')
print(f"📁 Latest training run: {latest_run}")

# 1. Show Performance Metrics Graph
results_path = os.path.join(latest_run, 'results.png')
print(f"Checking for results at: {results_path}")
print(f"Results file exists: {os.path.exists(results_path)}")

if os.path.exists(results_path):
    print("✅ Loading results image...")
    plt.figure(figsize=(15, 10))
    img = plt.imread(results_path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("⚠️ Results image not found")

# 2. Run Detection on a Test Image
best_weights = os.path.join(latest_run, 'weights', 'best.pt')
print(f"\nChecking for weights at: {best_weights}")
print(f"Weights file exists: {os.path.exists(best_weights)}")

if os.path.exists(best_weights):
    print("✅ Loading trained model...")
    trained_model = YOLO(best_weights)

    # Grab one image from the test folder
    test_img_path = str(project_dir / 'Tomato-Leaf-Disease-63' / 'test' / 'images')
    print(f"Looking for test images at: {test_img_path}")
    print(f"Test path exists: {os.path.exists(test_img_path)}")

    if os.path.exists(test_img_path):
        sample_images = [os.path.join(test_img_path, f) for f in os.listdir(test_img_path) if f.endswith(('.jpg', '.png'))]
        print(f"Found {len(sample_images)} test images")

        if sample_images:
            print(f"✅ Detecting diseases in: {sample_images[0]}")
            results = trained_model.predict(source=sample_images[0], conf=0.25)

            # Display the result with bounding boxes in the notebook
            for r in results:
                im_array = r.plot()  # plot a BGR numpy array of predictions
                plt.figure(figsize=(10, 10))
                plt.imshow(im_array[..., ::-1]) # Convert BGR to RGB
                plt.axis('off')
                plt.show()
        else:
            print("⚠️ No test images found")
    else:
        print("⚠️ Test image path does not exist")
else:
    print("⚠️ Trained weights not found. Please train a model first.")


In [ ]:
# ============================================================
# 🔼 UPLOAD YOUR OWN IMAGES FOR DISEASE DETECTION
# ============================================================
# Run this cell to select any number of images from your computer.
# A file dialog will open — select one or multiple images (.jpg, .png, .jpeg, .bmp, .webp).
# The model will detect diseases on each uploaded image and display results.

import tkinter as tk
from tkinter import filedialog
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# Load the trained model
project_dir = Path(r'C:\Users\revan\PyCharmMiscProject\Project')
run_dir = project_dir / 'runs' / 'detect'
matching = sorted(run_dir.glob('tomato_disease_notebook*'), key=os.path.getmtime) if run_dir.exists() else []
latest_run = str(matching[-1]) if matching else str(run_dir / 'tomato_disease_notebook')
best_weights = os.path.join(latest_run, 'weights', 'best.pt')

if not os.path.exists(best_weights):
    print("❌ Trained weights not found. Please train a model first.")
else:
    model = YOLO(best_weights)
    print(f"✅ Model loaded from: {best_weights}")

    # Open file dialog to select images
    root = tk.Tk()
    root.withdraw()  # Hide the main tkinter window
    root.attributes('-topmost', True)  # Bring dialog to front

    file_paths = filedialog.askopenfilenames(
        title="Select Test Images for Disease Detection",
        filetypes=[
            ("Image files", "*.jpg *.jpeg *.png *.bmp *.webp"),
            ("All files", "*.*")
        ]
    )
    root.destroy()

    if not file_paths:
        print("⚠️ No images selected.")
    else:
        print(f"\n📂 {len(file_paths)} image(s) selected for detection:\n")

        for idx, img_path in enumerate(file_paths, 1):
            print(f"{'='*60}")
            print(f"🖼️  Image {idx}/{len(file_paths)}: {os.path.basename(img_path)}")
            print(f"{'='*60}")

            # Run prediction
            results = model.predict(source=img_path, conf=0.25, verbose=False)

            for r in results:
                # Print detection summary
                if len(r.boxes) == 0:
                    print("   ⚠️ No diseases detected in this image.")
                else:
                    print(f"   ✅ {len(r.boxes)} detection(s) found:")
                    for box in r.boxes:
                        cls_id = int(box.cls[0])
                        cls_name = r.names[cls_id]
                        confidence = float(box.conf[0])
                        print(f"      • {cls_name}: {confidence:.1%} confidence")

                # Display image with bounding boxes
                im_array = r.plot()
                plt.figure(figsize=(10, 8))
                plt.imshow(im_array[..., ::-1])  # BGR to RGB
                plt.axis('off')
                plt.title(f"Image {idx}: {os.path.basename(img_path)}", fontsize=14)
                plt.tight_layout()
                plt.show()

            print()

        print(f"\n{'='*60}")
        print(f"✅ Detection complete! Processed {len(file_paths)} image(s).")
        print(f"{'='*60}")


✅ Model loaded from: C:\Users\revan\PyCharmMiscProject\Project\runs\detect\tomato_disease_notebook4\weights\best.pt
